In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l5.3 Learning-rate schedules

A fixed learning rate is rarely best. Warm it up (avoid early instability while
the moments are still cold), then cosine-decay it down (anneal into a fine,
settled minimum).

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pocketlm import cosine_lr

total = 200
lrs = [cosine_lr(s, warmup=20, total=total, base_lr=3e-4) for s in range(total)]
plt.plot(lrs)
plt.xlabel("step")
plt.ylabel("learning rate")
plt.title("warmup + cosine decay")
plt.show()
print("start:", round(lrs[0], 7), " peak:", round(lrs[19], 7),
      " end:", round(lrs[-1], 8))

The shape is a ramp then a gentle slope to near zero. This single schedule covers
the vast majority of from-scratch training runs.

In [ ]:
assert lrs[0] < lrs[19]      # warming up
assert lrs[-1] < lrs[19]     # decaying